# KlaDB human development: local convolution and concatenation fusion

PCBert-Kla (`baseline`), MM-Kla with K-centered multi-scale convolutions and concatenation fusion (`token_gated_conv_concat`), and an architecture-aligned HybridKla implementation are trained on the same locked human training partition and compared on its validation partition. Relative to the original token-gated model, the new model makes two changes: (1) depthwise-separable 1D convolutions of widths 3, 5, and 9 around the central lysine augment the attention-pooled sequence representation; (2) projected sequence and 27-feature representations are concatenated and mapped to the classifier input instead of being combined by a weighted-average gate. The feature projection and residual classifier are unchanged.

This is **development-only**: neither the internal test nor the study-held-out external test is scored. HybridKla retains its eight-branch, grouped out-of-fold stack; it is not an exact reproduction of the authors' training code. The two ProtBERT models use matched AdamW settings. HybridKla uses its own branch-specific optimizers and training schedule, so this compares complete pipelines on common data rather than architectures under identical optimization.


## 1. Run configuration

Run in Kaggle with Internet enabled, a two-T4 GPU accelerator, a `GITHUB_TOKEN` Kaggle Secret for the private code repository, and one private input dataset containing the adjacent `split_audit.json` and `model_manifest.csv` from `outputs/kladb_human_locked`. The raw KlaDB CSV is not needed when this locked pair is attached. `check` verifies inputs; `pilot` times one HybridKla ESM2 fold for one epoch; `development` trains all three models for **one selected seed**. Save separate notebook versions for seeds 42, 123, and 777. Run the pilot first to assess the runtime; one seed is not guaranteed to finish within a Kaggle session.

In [ ]:
from pathlib import Path

RUN_MODE = 'check'  # 'check', 'pilot', or 'development'
DECLARED_SEEDS = (42, 123, 777)
RUN_SEED = 42  # Change to 123 or 777 for a separate saved run.
EXPECTED_COMMIT = None  # Pin later seeds to the first run's runtime.json git_commit.
LOCK_DIR = None  # Set an explicit /kaggle/input/... directory if multiple locks are mounted.
LOCAL_HYBRIDKLA_ARCHIVE = None  # Optional path to an attached official HybridKla.zip.

ROOT = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
REPO = ROOT / 'MM-Kla'
PROJECT = REPO / 'experiment' / 'pcbert_kla_clean'
OUTPUT = ROOT / 'kla_human_conv_concat_development'
REPO_URL = 'https://github.com/kaleabnega/MM-Kla.git'
if RUN_MODE not in {'check', 'pilot', 'development'} or RUN_SEED not in DECLARED_SEEDS:
    raise ValueError('Invalid run mode or seed')
if not ROOT.is_dir() or not INPUT.is_dir():
    raise RuntimeError('This notebook requires a Kaggle runtime')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Run mode:', RUN_MODE, 'Seed:', RUN_SEED)

## 2. Code and environment

The checkout commit is recorded in each result bundle. Set `EXPECTED_COMMIT` to the first run's full commit hash for later seeds. A pinned run clones the full history and checks out that commit before training; an existing shallow checkout without it requires a fresh Kaggle session. Git receives the secret through a temporary askpass script; the token is never placed in the URL, notebook output, or saved run metadata.

In [ ]:
import hashlib, importlib.metadata, json, os, subprocess, sys, tempfile, time, urllib.request, zipfile

if not REPO.exists():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Kaggle Secret GITHUB_TOKEN is missing')
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / 'askpass.sh'
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf "%s\\n" x-access-token ;;\n  *Password*) printf "%s\\n" "$MM_KLA_GITHUB_TOKEN" ;;\n  *) exit 1 ;;\nesac\n')
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'MM_KLA_GITHUB_TOKEN': token})
        try:
            clone_command = ['git', '-c', 'credential.helper=', 'clone', '--branch', 'main']
            if EXPECTED_COMMIT is None:
                clone_command.extend(['--depth', '1'])
            clone_command.extend([REPO_URL, str(REPO)])
            clone = subprocess.run(clone_command, env=env, capture_output=True, text=True)
        finally:
            del token, env
    if clone.returncode:
        raise RuntimeError('Private clone failed; verify Internet, token access, and repository permissions')
elif not (REPO / '.git').is_dir():
    raise RuntimeError('Existing repository path is not a Git checkout')
if EXPECTED_COMMIT is not None:
    if not isinstance(EXPECTED_COMMIT, str) or len(EXPECTED_COMMIT) != 40 or any(character not in '0123456789abcdef' for character in EXPECTED_COMMIT):
        raise ValueError('EXPECTED_COMMIT must be a full lowercase 40-character Git commit hash')
    available = subprocess.run(['git', 'cat-file', '-e', EXPECTED_COMMIT + '^{commit}'], cwd=REPO, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if available.returncode:
        raise RuntimeError('Pinned commit is not in this checkout; start a fresh Kaggle session')
    subprocess.run(['git', 'checkout', '--detach', EXPECTED_COMMIT], cwd=REPO, check=True, capture_output=True, text=True)
if not (PROJECT / 'scripts/run_hybridkla_architecture.py').is_file():
    raise RuntimeError('The HybridKla architecture runner is missing; use a fresh checkout')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=PROJECT, check=True)
help_text = subprocess.check_output([sys.executable, 'scripts/run_hybridkla_architecture.py', '--help'], cwd=PROJECT, text=True)
if '--development-only' not in help_text:
    raise RuntimeError('Checkout predates the validation-only HybridKla runner; use a fresh Kaggle session')
revision_help = subprocess.check_output([sys.executable, 'scripts/run_replication.py', '--help'], cwd=PROJECT, text=True)
if 'token_gated_conv_concat' not in revision_help:
    raise RuntimeError('Checkout predates the revised model; use a fresh Kaggle session')
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
if EXPECTED_COMMIT is not None and COMMIT != EXPECTED_COMMIT:
    raise RuntimeError(f'Checkout commit {COMMIT} differs from expected {EXPECTED_COMMIT}')
import torch
if RUN_MODE != 'check' and torch.cuda.device_count() < 2:
    raise RuntimeError('Training is configured for a Kaggle 2 x T4 GPU runtime')
runtime = {'git_commit': COMMIT, 'seed': RUN_SEED, 'mode': RUN_MODE, 'python': sys.version.split()[0], 'torch': torch.__version__, 'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
(OUTPUT / 'runtime.json').write_text(json.dumps(runtime, indent=2, sort_keys=True) + '\n')
print(json.dumps(runtime, indent=2))

## 3. Locked human split

Both source-file hashes must match the uncapped KlaDB human v1 benchmark. Training uses 9,970 positives and 29,910 sampled putative negatives; validation keeps 1,436 positives and all 16,277 putative negatives. The independent partitions are verified but are not evaluated here.

In [ ]:
AUDIT_SHA = '2d1275577a430ff5758aa89c01a62c138c294ddc537bc67f671db3807856a0b8'
MANIFEST_SHA = '842ff1b767d60f5645d3ba9c93150ec42ed9966dcc13672a679269327a25b95e'
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

if LOCK_DIR is None:
    candidates = [p.parent for p in INPUT.rglob('split_audit.json') if (p.parent / 'model_manifest.csv').is_file()]
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one paired lock dataset, found {len(candidates)}; set LOCK_DIR')
    LOCK_DIR = candidates[0]
LOCK_DIR = Path(LOCK_DIR)
LOCK = LOCK_DIR / 'split_audit.json'
MANIFEST = LOCK_DIR / 'model_manifest.csv'
if sha256(LOCK) != AUDIT_SHA or sha256(MANIFEST) != MANIFEST_SHA:
    raise ValueError('Attached files do not match the locked KlaDB human v1 benchmark')
audit = json.loads(LOCK.read_text())
if audit.get('train_positive_cap') != 0 or audit.get('sampled_training_rows') != 39880 or audit.get('holdout_pmid') != '41192556':
    raise ValueError('Attached benchmark is not the uncapped human study split')
subprocess.run([sys.executable, 'scripts/run_replication.py', '--run', 'human-study-check', '--human-lock', str(LOCK)], cwd=PROJECT, check=True)
print('Verified lock:', LOCK_DIR)

## 4. HybridKla resources

The architecture-aligned runner uses `top10.txt` and `BLOSUM62R.txt` from the official HybridKla archive. The archive is checksum-verified before extraction. A mounted copy can be specified in the configuration cell; otherwise the source is downloaded when `pilot` or `development` is selected. No published training rows are added to the KlaDB benchmark.

In [ ]:
RESOURCE_DIR = OUTPUT / 'hybridkla_resources'
if RUN_MODE in {'pilot', 'development'}:
    archive_path = Path(LOCAL_HYBRIDKLA_ARCHIVE) if LOCAL_HYBRIDKLA_ARCHIVE else OUTPUT / 'HybridKla.zip'
    if not archive_path.is_file() and LOCAL_HYBRIDKLA_ARCHIVE is None:
        urllib.request.urlretrieve('https://transkla.zzu.edu.cn/api/download/HybridKla.zip', archive_path)
    if not archive_path.is_file() or sha256(archive_path) != '73ddb128c441d7b213c4636e6bb6ce34501e82143c2b9bc06ec329d68a422993':
        raise ValueError('Official HybridKla archive is missing or has changed')
    RESOURCE_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for filename in ('top10.txt', 'BLOSUM62R.txt'):
            (RESOURCE_DIR / filename).write_bytes(archive.read('HybridKla/train/' + filename))
    print('Verified HybridKla resources:', RESOURCE_DIR)
else:
    print('Resource preparation skipped in check mode')

## 5. Development runs

PCBert-Kla and MM-Kla-revised run concurrently, one per T4: four trainable encoder layers, AdamW (`2e-5`, weight decay `0.01`), linear schedule with 10% warmup, batch size 4, 30-epoch maximum, patience 5, no positive weighting, checkpoint selection by validation AUPRC. HybridKla then runs on one GPU with five grouped out-of-fold branches per feature type and its architecture-specific training settings. Its pilot is a one-fold, one-epoch timing check, **not** a performance estimate.

The revised model differs from the original token-gated model in both local sequence modelling and fusion. Compare its validation results with the separately recorded convolution-only run to isolate the incremental fusion change; the three models in this notebook alone cannot isolate that effect. The three pipelines share the same train/validation rows and seed. HybridKla's optimization is not AdamW throughout and its training budget is not identical to ProtBert's; these differences must be disclosed in a manuscript.

In [ ]:
def run_logged(command, destination, label):
    if destination.exists():
        raise RuntimeError(f'Refusing to overwrite prior {label} output: {destination}')
    destination.mkdir(parents=True)
    log_path = destination / 'training.log'
    started = time.monotonic()
    with log_path.open('w') as log:
        process = subprocess.Popen(command, cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT)
        try:
            while process.poll() is None:
                time.sleep(60)
                lines = log_path.read_text(errors='replace').splitlines()
                print(label, lines[-1] if lines else 'running', flush=True)
        except BaseException:
            process.terminate()
            process.wait()
            raise
    if process.returncode:
        raise RuntimeError(f'{label} failed (exit {process.returncode}):\n' + '\n'.join(log_path.read_text(errors='replace').splitlines()[-30:]))
    print(label, 'completed in', round(time.monotonic() - started, 1), 's')
    return round(time.monotonic() - started, 1)

def protbert_command(architecture, device, destination, seed):
    return [sys.executable, '-u', 'scripts/run_replication.py', '--run', 'human-study-development', '--human-lock', str(LOCK), '--architecture', architecture, '--model-name', 'Rostlab/prot_bert', '--encoder-layers', '4', '--epochs', '30', '--patience', '5', '--batch-size', '4', '--device', device, '--optimizer', 'adamw', '--learning-rate', '2e-5', '--weight-decay', '0.01', '--scheduler', 'linear', '--warmup-ratio', '0.1', '--positive-weighting', 'none', '--monitor-metric', 'AUPRC', '--decision-threshold', '0.5', '--seed', str(seed), '--output-dir', str(destination)]

def hybrid_command(destination, seed, pilot=False):
    command = [sys.executable, '-u', 'scripts/run_hybridkla_architecture.py', '--benchmark-lock', str(LOCK), '--resource-dir', str(RESOURCE_DIR), '--output-dir', str(destination), '--device', 'cuda:0', '--seed', str(seed), '--esm-model', 'facebook/esm2_t30_150M_UR50D', '--esm-epochs', '40', '--lstm-epochs', '30', '--feature-epochs', '100', '--meta-epochs', '50', '--esm-batch-size', '32', '--lstm-batch-size', '32', '--feature-batch-size', '2048', '--patience', '6']
    command.append('--pilot' if pilot else '--development-only')
    return command

def run_seed(seed):
    seed_dir = OUTPUT / f'seed_{seed}'
    active = {}
    try:
        for architecture, device in (('baseline', 'cuda:0'), ('token_gated_conv_concat', 'cuda:1')):
            destination = seed_dir / architecture
            if destination.exists():
                raise RuntimeError(f'Refusing to overwrite prior output: {destination}')
            destination.mkdir(parents=True)
            log_path = destination / 'training.log'
            log = log_path.open('w')
            process = subprocess.Popen(protbert_command(architecture, device, destination, seed), cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT)
            active[architecture] = (process, log, log_path)
            print('Started', architecture, 'on', device, flush=True)
        while active:
            for architecture, (process, log, log_path) in list(active.items()):
                if process.poll() is None:
                    continue
                log.close()
                if process.returncode:
                    raise RuntimeError(f'{architecture} failed (exit {process.returncode}):\n' + '\n'.join(log_path.read_text(errors='replace').splitlines()[-30:]))
                print(architecture, 'completed', flush=True)
                del active[architecture]
            if active:
                time.sleep(60)
                for architecture, (_, _, log_path) in active.items():
                    lines = log_path.read_text(errors='replace').splitlines()
                    print(architecture, lines[-1] if lines else 'running', flush=True)
    finally:
        for process, log, _ in active.values():
            if process.poll() is None:
                process.terminate()
                process.wait()
            log.close()
    hybrid_dir = seed_dir / 'hybridkla_architecture_aligned'
    run_logged(hybrid_command(hybrid_dir, seed), hybrid_dir, f'HybridKla seed {seed}')

if RUN_MODE == 'pilot':
    pilot_seed = RUN_SEED
    pilot_dir = OUTPUT / f'seed_{pilot_seed}' / 'hybridkla_pilot'
    run_logged(hybrid_command(pilot_dir, pilot_seed, pilot=True), pilot_dir, 'HybridKla pilot')
    pilot_report = json.loads((pilot_dir / 'run_report.json').read_text())
    if pilot_report['mode'] != 'one-fold-training-only-timing' or pilot_report['test_metrics']:
        raise ValueError('Pilot unexpectedly scored a test partition')
    print('One-fold ESM2 pilot seconds:', pilot_report['fold_reports']['ESM2_fold_0']['wall_seconds'])
elif RUN_MODE == 'development':
    print('Development seed:', RUN_SEED, flush=True)
    run_seed(RUN_SEED)
else:
    print('Check complete; no model training or test inference was run')

## 6. Validation results

Metrics are read from the trained model outputs, not entered manually. The notebook verifies each run's split, seed, settings, and no-test mode before creating a three-row validation table, paired AUPRC differences, and a seed summary bundle. Three-seed means and standard deviations are calculated only after all three saved runs have completed. No test metric is read or written.

In [ ]:
import pandas as pd
from IPython.display import display

SETTINGS = {'mm_kla_architecture': 'token_gated_conv_concat', 'local_kernel_widths': [3, 5, 9], 'local_channels_per_width': 64, 'fusion_mode': 'concatenate_projected_branches', 'protbert_model': 'Rostlab/prot_bert', 'protbert_encoder_layers': 4, 'protbert_optimizer': 'adamw', 'protbert_learning_rate': 2e-5, 'protbert_weight_decay': 0.01, 'protbert_scheduler': 'linear', 'protbert_warmup_ratio': 0.1, 'protbert_epochs': 30, 'protbert_patience': 5, 'protbert_batch_size': 4, 'positive_weighting': 'none', 'hybrid_esm_model': 'facebook/esm2_t30_150M_UR50D', 'hybrid_esm_epochs': 40, 'hybrid_lstm_epochs': 30, 'hybrid_feature_epochs': 100, 'hybrid_meta_epochs': 50, 'hybrid_esm_batch_size': 32, 'hybrid_lstm_batch_size': 32, 'hybrid_feature_batch_size': 2048, 'hybrid_patience': 6, 'hybrid_folds': 5, 'selection_metric': 'AUPRC', 'threshold': 0.5}
METRICS = ('AUPRC', 'AUC', 'MCC', 'F1', 'Pre', 'Rec', 'SP', 'ACC')
if RUN_MODE == 'development':
    for seed in (RUN_SEED,):
        seed_dir = OUTPUT / f'seed_{seed}'
        records = []
        for architecture in ('baseline', 'token_gated_conv_concat'):
            directory = seed_dir / architecture
            matches = list(directory.rglob('validation_metrics.json'))
            if len(matches) != 1:
                raise RuntimeError(f'Expected one validation result for {architecture} seed {seed}, found {len(matches)}')
            config = json.loads(matches[0].with_name('run_config.json').read_text())
            expected = {'run': 'human-study-development', 'architecture': architecture, 'test_rows': 0, 'extra_test_rows': {}, 'model_seed': seed, 'human_lock_sha256': AUDIT_SHA, 'model_name': SETTINGS['protbert_model'], 'encoder_layers': 4, 'optimizer': 'adamw', 'learning_rate': 2e-5, 'weight_decay': 0.01, 'scheduler': 'linear', 'warmup_ratio': 0.1, 'epochs': 30, 'patience': 5, 'batch_size': 4, 'positive_weighting': 'none', 'monitor_metric': 'AUPRC', 'decision_threshold': 0.5, 'training_rows': 39880, 'validation_rows': 17713}
            if any(config.get(key) != value for key, value in expected.items()):
                raise ValueError(f'{architecture} seed {seed} crossed the development boundary')
            metrics = json.loads(matches[0].read_text())
            records.append({'seed': seed, 'architecture': architecture, 'selected_epoch': config['selected_epoch'], **{m: metrics[m] for m in METRICS}})
        hybrid_dir = seed_dir / 'hybridkla_architecture_aligned'
        hybrid = json.loads((hybrid_dir / 'run_report.json').read_text())
        hybrid_expected = {'esm_epochs': 40, 'lstm_epochs': 30, 'feature_epochs': 100, 'meta_epochs': 50, 'esm_batch_size': 32, 'lstm_batch_size': 32, 'feature_batch_size': 2048, 'patience': 6}
        if hybrid['mode'] != 'development-only' or hybrid['test_metrics'] or hybrid['benchmark_lock_sha256'] != AUDIT_SHA or hybrid['benchmark_manifest_sha256'] != MANIFEST_SHA or hybrid['seed'] != seed or hybrid['folds'] != 5 or hybrid['model_name'] != SETTINGS['hybrid_esm_model'] or hybrid['selection_metric'] != 'validation AUPRC' or hybrid['threshold'] != 0.5 or any(hybrid['hyperparameters'].get(key) != value for key, value in hybrid_expected.items()):
            raise ValueError(f'HybridKla seed {seed} crossed the development boundary')
        metrics = hybrid['validation_metrics']
        records.append({'seed': seed, 'architecture': 'hybridkla_architecture_aligned', 'selected_epoch': hybrid['fold_reports']['META']['selected_epoch'], **{m: metrics[m] for m in METRICS}})
        bundle = {'schema_version': 1, 'seed': seed, 'git_commit': COMMIT, 'audit_sha256': AUDIT_SHA, 'manifest_sha256': MANIFEST_SHA, 'settings': SETTINGS, 'records': records}
        bundle_path = OUTPUT / f'seed_{seed}_summary.json'
        bundle_path.write_text(json.dumps(bundle, indent=2, sort_keys=True) + '\n')
        print('Saved:', bundle_path)
    all_rows = pd.DataFrame(records)
    display(all_rows.sort_values(['seed', 'architecture']).round(4))
    paired = all_rows.pivot(index='seed', columns='architecture', values='AUPRC')
    paired['MM-Kla minus PCBert-Kla'] = paired['token_gated_conv_concat'] - paired['baseline']
    paired['MM-Kla minus HybridKla-aligned'] = paired['token_gated_conv_concat'] - paired['hybridkla_architecture_aligned']
    display(paired.round(4))
    all_rows.to_csv(OUTPUT / 'validation_results.csv', index=False)
    paired.to_csv(OUTPUT / 'validation_paired_auprc.csv')
else:
    print('No development results in check or pilot mode')

## Interpretation boundary

Validation AUPRC is the predeclared primary comparison measure and a model-selection measure, not independent performance. The three-model comparison should report paired seed differences and uncertainty, including the class prevalence of the validation set. HybridKla's architecture-aligned result must not be described as the authors' exact published implementation. The previously examined KlaDB external study cannot serve as a newly untouched confirmatory test for an optimizer chosen after seeing its results.

## 7. Shareable diagnostics

Each saved run produces one compact ZIP containing that seed's validation results, prediction CSVs, logs, and run metadata. Model checkpoints and the official resource archive are excluded. After downloading all three ZIPs, extract them to separate directories and run `python scripts/aggregate_human_adamw_seeds.py --bundles <seed_42_summary.json> <seed_123_summary.json> <seed_777_summary.json> --model-architecture token_gated_conv_concat --output-dir <directory>` from `experiment/pcbert_kla_clean`. The aggregation script verifies the seed set, code commit, dataset hashes, and settings before writing combined tables.

In [ ]:
if RUN_MODE in {'pilot', 'development'}:
    bundle_path = OUTPUT / (f'seed_{RUN_SEED}_development_diagnostics.zip' if RUN_MODE == 'development' else f'seed_{RUN_SEED}_hybridkla_pilot_diagnostics.zip')
    selected = [OUTPUT / 'runtime.json']
    if RUN_MODE == 'development':
        selected.extend([OUTPUT / 'validation_results.csv', OUTPUT / 'validation_paired_auprc.csv', OUTPUT / f'seed_{RUN_SEED}_summary.json'])
        model_dirs = [OUTPUT / f'seed_{RUN_SEED}' / architecture for architecture in ('baseline', 'token_gated_conv_concat', 'hybridkla_architecture_aligned')]
    else:
        model_dirs = [OUTPUT / f'seed_{RUN_SEED}' / 'hybridkla_pilot']
    for directory in model_dirs:
        selected.extend(path for path in directory.rglob('*') if path.is_file() and path.name in {'run_report.json', 'run_config.json', 'validation_metrics.json', 'validation_predictions.csv', 'training.log'})
    with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in selected:
            if not path.is_file():
                raise FileNotFoundError(path)
            archive.write(path, path.relative_to(OUTPUT))
    print('Saved diagnostics:', bundle_path, 'bytes:', bundle_path.stat().st_size)
else:
    print('No training diagnostics in check mode')